## Remote ID Sppoofing Attack in a City

In [ ]:
from simulator import Oracle, Simulator
from simulator.config import Color, Model
from simulator.entities import SimVehicle, SpoofProfile
from simulator.helpers import SimProcess, clean
from simulator.helpers.coordinates import ENU, ENUPose, GRAPose
from simulator.planner import AutoPlan
from simulator.visualizer import Gazebo, GazMarker

clean()

## Simulation Positions

In [ ]:
gra_origin = GRAPose(lat=-35.3633280, lon=149.1652241, alt=0, heading=0)
enu_origin = ENUPose(x=0, y=0, z=gra_origin.alt, heading=gra_origin.heading)

base_homes = ENUPose.list([(15, -10, 0, 0), (0, -15, 0, 0)])
base_paths = [
    ENU.list([(0, 0, 0), (0, 0, 5), (0, 25, 5)]),
    ENU.list([(0, 0, 0), (0, 0, 5), (30, 0, 5)]),
]


## Oracle

In [ ]:
orac = Oracle()

## Create Vehicles

In [ ]:
attacker_sysid = 2
sysids = [1, attacker_sysid]
colors = [Color.GREEN, Color.RED]
model = Model.IRIS
fake_pos = ENU(x=15, y=0, z=5)  # position the attacker broadcasts over Remote ID

for sysid, base_home, base_path, color in zip(
    sysids, base_homes, base_paths, colors, strict=True
):
    auto_plan = AutoPlan.from_relative_path(
        name="simple_auto_plan",
        sysid=sysid,
        gra_origin=gra_origin,
        relative_home=base_home,
        relative_path=base_path,
        navigation_speed=3.0,
        firmware=model.firmware,
    )

    veh = SimVehicle.from_relative(
        sysid=sysid,
        plan=auto_plan,
        color=color,
        enu_origin=enu_origin,
        relative_home=base_home,
        relative_path=base_path,
        model=model,
        avoidance=sysid != attacker_sysid,  # attacker ignores nearby traffic
        spoof=SpoofProfile.constant(fake_pos) if sysid == attacker_sysid else None,
    )
    orac.add_vehicle(veh)

## Scenario configurarion

In [ ]:
# Marker geometry for the Gazebo preview (fake_pos is set with the attacker above).
radar_radius = 10  # meters
safety_radius = 5  # meters

## Gazebo

In [ ]:
small_city_path = "simulator/visualizer/gazebo/worlds/small_city_demo.world"
gaz = Gazebo(gra_origin, world_path=small_city_path)

origin_gaz = GazMarker(
    name="origin", group="origin", pos=enu_origin.unpose(), color=Color.WHITE
)

fake_marker = GazMarker(
    name="fake_pos", group="fake_pos", pos=fake_pos, color=Color.ORANGE
)

avoid_zone = GazMarker(
    name="avoid_zone",
    group="avoid_zone",
    pos=fake_pos,
    color=Color.RED,
    radius=safety_radius,
    alpha=0.85,
)

for marker in [origin_gaz, fake_marker, avoid_zone]:
    gaz.markers.append(marker)


## Simulator

In [ ]:
simulator = Simulator(
    oracle=orac,
    visualizer=gaz,
    terminals=[SimProcess.LOGIC],
    verbose=1,
)

simulator.preview()


## Run

In [ ]:
simulator.run()

## Ground truth: real vs transmitted

The Oracle only *receives* Remote ID, which the attacker (sysid 255) spoofs, so
its live track is the **transmitted** position (dots). `truth=` overlays the
**real** trajectory (line), reconstructed by the Oracle from the ground-truth
logs — the spoof is the gap between the two.

### Transmitted only (spoofed) — the Oracle's raw Remote ID view

In [ ]:
orac.plot_trajectories();

### Real path (line) overlaid on the transmitted RID (dots)

In [ ]:
orac.plot_trajectories(truth=True);

### Just the attacker's real vs spoofed position


In [ ]:
orac.plot_trajectories(truth=[attacker_sysid]);